# Import libraries

In [92]:
import pandas as pd
import time
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

True

# LLMs

In [79]:
class DefaultResponse(BaseModel):
    response: str

In [91]:
class Bielik:
    def __init__(self):
        self.__base_url = "https://153.19.237.85/api/llm/prompt/chat"
        auth = (os.getenv("LLM_USERNAME"), os.getenv("LLM_PASSWORD"))
        self.__auth_kwargs = {
            'auth': auth,
            'verify': False, # Disable SSL verification
        }

    def generate_response(self, system_prompt, user_prompt):
        data = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "max_length": 8192,  # adjust as needed
            "temperature": 0.0,
        }

        response = requests.put(
            self.__base_url,
            json=data,
            headers={
                'Accept': 'application/json',
                'Content-Type': 'application/json'
            },
            **self.__auth_kwargs,
        )
        response.raise_for_status()
        response_json = response.json()

        return response_json

In [127]:
class GPT:
    def __init__(self):
        self.__client = OpenAI()

    def generate_response(self, system_prompt, user_prompt, output_schema=DefaultResponse):
        response = self.__client.responses.parse(
            model="gpt-4.1",
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            text_format=output_schema,
        )

        return response.output_parsed


# Prompts

In [115]:
class FactCheck(BaseModel):
    statement: str = Field(
        description="Pojedyncze, wyizolowane zdanie lub stwierdzenie wyciągnięte z uzasadnienia."
    )
    is_true: bool = Field(
        description="Czy to konkretne stwierdzenie jest w 100% zgodne z prawdą historyczną, naukową lub logiczną?"
    )
    error_explanation: str | None = Field(
        description="Jeśli is_true to False, napisz krótko, na czym polega błąd. Jeśli True, zostaw null/puste."
    )

class EvaluationResponse(BaseModel):
    extracted_facts: list[FactCheck] = Field(
        description="Rozbij całe uzasadnienie na pojedyncze, atomowe fakty i oceń prawdziwość każdego z nich z osobna."
    )
    is_answer_correct: bool = Field(
        description=(
            "Zwróć True TYLKO wtedy, gdy oceniana odpowiedź końcowa jest merytorycznie "
            "w 100% zgodna z wzorcem. Zwróć False, jeśli występuje jakiekolwiek odstępstwo, "
            "błąd lub jeśli odpowiedź jest tylko częściowo poprawna."
        )
    )
    is_reasoning_correct: bool = Field(
        description=(
            "Zwróć True TYLKO ORAZ WYŁĄCZNIE, jeśli uzasadnienie jest absolutnie bezbłędne "
            "zarówno pod kątem logicznym, jak i FAKTOGRAFICZNYM. Wszystkie użyte w nim daty, liczby, "
            "nazwy i fakty muszą być w 100% prawdziwe. "
            "Zwróć BEZWZGLĘDNIE False, jeśli w uzasadnieniu pojawi się JAKIKOLWIEK błąd rzeczowy "
            "(np. błędny rok), fałszywe założenie, halucynacja lub zgadywanie, nawet jeśli "
            "końcowy wynik jest poprawny. "
            "Jedyny wyjątek, kiedy zwracasz True pomimo błędu, to omyłka polegająca na złym przepisaniu "
            "poprawnego wyniku z uzasadnienia do ostatecznej odpowiedzi."
        )
    )

calibrated_reasoning_prompt = """
Jesteś niezwykle surowym ewaluatorem faktów i logiki.
Twoim zadaniem jest ocena uzasadnienia i odpowiedzi końcowej.

ZASADA ZERO TOLERANCJI: Wymagam absolutnej precyzji. Najpierw musisz rozbić uzasadnienie na pojedyncze fakty i zweryfikować każdy z nich. Wystąpienie choćby jednego błędnego faktu (np. fałszywa data, zła liczba) bezwzględnie dyskwalifikuje całe uzasadnienie.

### DANE WEJŚCIOWE:
Pytanie: {question}
Wzorzec (Poprawna odpowiedź): {correct_answer}
Wygenerowane uzasadnienie: {reason}

### ZADANIE:
Wypełnij schemat. Pamiętaj: najpierw rozbij tekst na fakty i bezlitośnie je zweryfikuj. Jeśli jakikolwiek wyciągnięty fakt jest błędny, główna flaga 'is_reasoning_correct' musi wynosić False.
"""

In [116]:
medical_system_prompt = f"""
Jesteś wybitnym lekarzem i ekspertem medycznym o encyklopedycznej wiedzy.
Twoim zadaniem jest rozwiązywanie pytań testowych krok po kroku. Twój proces myślowy musi być precyzyjny: przeanalizuj każdą dostępną opcję przed wydaniem ostatecznego werdyktu.
"""

medical_user_prompt = """
Rozwiąż poniższe pytanie medyczne, opierając się na dostarczonych informacjach.

### DODATKOWE DANE / OPIS PRZYPADKU:
{data}

### TREŚĆ PYTANIA I DOSTĘPNE OPCJE:
{question}
"""

# Define metrics

In [75]:
def measure_response_time(function, *args, **kwargs):
    start = time.perf_counter()
    result = function(*args, **kwargs)
    end = time.perf_counter()

    return end - start, result

In [10]:
def count_words(text):
    text = text.split()

    return len(text)

In [117]:
def calibrated_reasoning(question: str, correct_answer: str, reason: str) -> EvaluationResponse:
    gpt = GPT()

    prompt = calibrated_reasoning_prompt.format(
        question=question,
        correct_answer=correct_answer,
        reason=reason,
    )

    response = gpt.generate_response(prompt, "", EvaluationResponse)
    return response

# Generate answers

In [134]:
data = pd.read_json("data/lek_pl_sample.json")
data.drop(["edition", "year", "season", "question_id"], axis=1, inplace=True)
data.reset_index(inplace=True, drop=True)
data.rename(columns={"question_w_options": "question"}, inplace=True)

OUTPUT_FILE = "data/results.jsonl"

def save_result(record, file_path=OUTPUT_FILE):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


model = Bielik()

for i in range(len(data)):
    question = data.iloc[i]["question"]
    correct_answer = data.iloc[i]["answer"]

    response_time, response = measure_response_time(
        model.generate_response,
        medical_system_prompt,
        medical_user_prompt.format(question=question, data="")
    )

    text = response.get("response", "")
    used_words = count_words(text)

    score = calibrated_reasoning(question, correct_answer, text)

    print(f"Pytanie: {i}")
    print(text)
    print(score)

    is_answer_correct = float(score.is_answer_correct)
    is_reasoning_correct = float(score.is_reasoning_correct)

    facts = score.extracted_facts
    fact_accuracy = sum(f.is_true for f in facts) / len(facts) if facts else 0.0

    all_facts = [
        {
            "statement": f.statement,
            "is_true": f.is_true,
            "error_explanation": f.error_explanation
        }
        for f in score.extracted_facts
    ]

    record = {
        "question": question,
        "correct_answer": correct_answer,
        "response": text,
        "response_time": response_time,
        "used_words": used_words,
        "is_answer_correct": is_answer_correct,
        "is_reasoning_correct": is_reasoning_correct,
        "fact_accuracy": fact_accuracy,
        "all_facts": all_facts,
    }

    save_result(record)



/home/quuixly/Agent-Orchestrated-Hybrid-RAG/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '153.19.237.85'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Pytanie: 0
Aby rozwiązać to pytanie, musimy najpierw zrozumieć, czym są nowotwory germinalne jajnika i jakie typy nowotworów do nich należą.

Nowotwory germinalne jajnika to nowotwory wywodzące się z komórek zarodkowych (germinalnych). Są one zazwyczaj łagodne, ale mogą być również złośliwe.

Przeanalizujmy każdą z opcji:

A. Rozrodczak (Sertoli-Leydig cell tumor) - Jest to łagodny nowotwór wywodzący się z komórek zarodkowych, ale nie jest klasyfikowany jako nowotwór germinalny.

B. Potworniak (Teratoma) - Jest to nowotwór germinalny, który może być łagodny lub złośliwy.

C. Rak embrionalny (Embryonal carcinoma) - Jest to złośliwy nowotwór germinalny.

D. Ziarniszczak (Granulosa cell tumor) - Jest to łagodny nowotwór wywodzący się z komórek zarodkowych, ale nie jest klasyfikowany jako nowotwór germinalny.

E. Rak kosmówki (Choriocarcinoma) - Jest to złośliwy nowotwór wywodzący się z trofoblastu, a nie z komórek zarodkowych, więc nie jest klasyfikowany jako nowotwór germinalny.

### Ana

/home/quuixly/Agent-Orchestrated-Hybrid-RAG/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '153.19.237.85'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Pytanie: 1
### Analiza pytania i dostępnych opcji:

1. Zdanie 1: "Podstawowym lekiem stosowanym w POChP są glikokortykosteroidy wziewne."
   - Glikokortykosteroidy wziewne są rzeczywiście jednym z podstawowych leków stosowanych w leczeniu przewlekłej obturacyjnej choroby płuc (POChP).

2. Zdanie 2: "Ponieważ hamują postępujące upośledzenie czynności płuc."
   - Glikokortykosteroidy wziewne mają działanie przeciwzapalne, co pomaga w kontrolowaniu stanu zapalnego w drogach oddechowych, co jest jednym z kluczowych mechanizmów prowadzących do postępującego upośledzenia czynności płuc w POChP.

### Ocena związku przyczynowego:
- Glikokortykosteroidy wziewne są stosowane w leczeniu POChP, ponieważ mają zdolność hamowania postępującego upośledzenia czynności płuc poprzez swoje działanie przeciwzapalne.

### Wybór odpowiedzi:
- Opcja A: "obie informacje są prawdziwe i istnieje między nimi związek przyczynowy."

### Podsumowanie:
Oba zdania są prawdziwe i istnieje między nimi związek przyczynow

/home/quuixly/Agent-Orchestrated-Hybrid-RAG/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '153.19.237.85'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

Pytanie: 397
Aby rozwiązać to pytanie, musimy przeanalizować każdą opcję i sprawdzić, która z nich najlepiej opisuje prawidłowy kardiotokograficzny test niestresowy w ciąży.

1. Podstawowa czynność serca płodu - 100/min:
   - Wartość 100 uderzeń na minutę (bpm) jest zbyt niska dla prawidłowej czynności serca płodu. Zazwyczaj prawidłowa czynność serca płodu wynosi około 120-160 bpm.

2. Podstawowa czynność serca płodu - 150/min:
   - Ta wartość jest w normie i mieści się w zakresie prawidłowej czynności serca płodu.

3. Skacząca zmienność czynności serca płodu:
   - Skacząca zmienność (tzw. "skacząca" lub "zmienna" czynność serca) jest często obserwowana w prawidłowym kardiotokograficznym teście niestresowym i jest uważana za normalną.

4. Brak akceleracji:
   - Akceleracja to nagłe zwiększenie częstości akcji serca płodu, które jest często obserwowane w prawidłowym kardiotokograficznym teście niestresowym. Brak akceleracji może wskazywać na nieprawidłowości.

5. Brak deceleracji:
   - 

/home/quuixly/Agent-Orchestrated-Hybrid-RAG/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '153.19.237.85'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Pytanie: 398
Aby rozwiązać to pytanie, przeanalizujmy każdą opcję krok po kroku:

A. W sytuacji ogłoszenia stanu epidemii.
- Stan epidemii może wpływać na dostępność zasobów medycznych i możliwości leczenia, ale nie jest to bezpośredni powód, aby ograniczać informacje pacjentowi.

B. Na żądanie osoby bliskiej pacjenta, jeśli pacjent jest pełnoletni, przytomny i zdolny do zrozumienia znaczenia informacji, ale taka jest wola osoby bliskiej.
- Zgodnie z prawem, pacjent ma prawo do pełnej informacji o swoim stanie zdrowia, chyba że wyraźnie zrzeknie się tego prawa. Żądanie osoby bliskiej nie jest wystarczającym powodem do ograniczenia informacji.

C. Na żądanie Rzecznika Praw Lekarza, działającego przy Okręgowej Izbie Lekarskiej.
- Rzecznik Praw Lekarza nie ma uprawnień do decydowania o tym, jakie informacje medyczne mogą być udzielane pacjentowi.

D. Ze względu na udział pacjenta w badaniu klinicznym.
- Udział w badaniu klinicznym może wiązać się z ograniczeniem informacji, ale tylko wted

/home/quuixly/Agent-Orchestrated-Hybrid-RAG/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '153.19.237.85'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Pytanie: 399
Aby rozwiązać to pytanie, przeanalizujmy każdą opcję krok po kroku, opierając się na aktualnej wiedzy medycznej:

A. najczęściej występuje przed 40. rokiem życia.
- Rak sromu jest rzadkim nowotworem, który najczęściej występuje u kobiet po menopauzie.
- Większość przypadków diagnozowana jest u kobiet w wieku 60-70 lat.
- Zatem ta opcja jest fałszywa.

B. stanowi 3-8% wszystkich nowotworów złośliwych żeńskich narządów płciowych.
- Rak sromu jest stosunkowo rzadkim nowotworem, stanowiącym około 3-8% wszystkich nowotworów złośliwych żeńskich narządów płciowych.
- Ta opcja jest prawdziwa.

C. ponad 90% raków sromu stanowią raki gruczołowe.
- Rak sromu może przybierać różne formy, w tym raki gruczołowe, płaskonabłonkowe i inne.
- Raki gruczołowe stanowią około 50-60% przypadków, a nie ponad 90%.
- Zatem ta opcja jest fałszywa.

D. leczeniem z wyboru jest leczenie systemowe.
- Leczenie raka sromu zazwyczaj obejmuje chirurgię, radioterapię i chemioterapię, w zależności od stadium

In [138]:
dat = pd.read_json("data/results.jsonl", lines=True)

In [139]:
dat

,question,correct_answer,response,response_time,used_words,is_answer_correct,is_reasoning_correct,fact_accuracy,all_facts
0,Do nowotworów germinalnych jajnika nie należy:...,D,"Aby rozwiązać to pytanie, musimy najpierw zroz...",30.421301,217,0,0,0.416667,[{'statement': 'Nowotwory germinalne jajnika t...
1,Podstawowym lekiem stosowanym w POChP są gliko...,E,### Analiza pytania i dostępnych opcji:\n\n1. ...,24.871469,158,0,0,0.400000,[{'statement': 'Glikokortykosteroidy wziewne s...
2,Niewydolność cieśniowo-szyjkowa w wywiadzie je...,A,"Aby rozwiązać to pytanie, przeanalizujmy każdą...",34.917844,229,1,1,1.000000,[{'statement': 'Niewydolność cieśniowo-szyjkow...
3,Nazwę „schizofrenia” wprowadził:\nA. Kraepelin...,E,"Aby rozwiązać to pytanie, musimy zidentyfikowa...",26.597869,182,0,0,0.600000,[{'statement': 'Emil Kraepelin był niemieckim ...
4,W ciągu ostatniego roku zakażenie Legionella p...,A,"Aby rozwiązać to pytanie, przeanalizujmy każde...",32.150393,231,0,0,0.700000,[{'statement': 'Legionelloza u małych dzieci j...
...,...,...,...,...,...,...,...,...,...
395,Zgodnie z aktualnie obowiązującym Programem Sz...,C,"Aby rozwiązać to pytanie, musimy dokładnie prz...",35.590993,249,0,0,0.727273,[{'statement': 'Personel medyczny jest narażon...
396,Prawidłowy wynik konwencjonalnej radiografii k...,A,"Aby rozwiązać to pytanie, przeanalizujmy obie ...",41.195588,277,1,1,1.000000,[{'statement': 'Prawidłowy wynik radiografii k...
397,Prawidłowy kardiotokograficzny test niestresow...,C,"Aby rozwiązać to pytanie, musimy przeanalizowa...",42.260359,292,0,0,0.625000,[{'statement': 'Wartość 100 uderzeń na minutę ...
398,"Wskaż, kiedy lekarz onkolog może ograniczyć in...",E,"Aby rozwiązać to pytanie, przeanalizujmy każdą...",32.074334,255,1,0,0.833333,[{'statement': 'Stan epidemii może wpływać na ...
